# Плотное векторное представление слов для определения тональности текста отзывов на фильмы из IMDb (Internet Movie Database) — PyTorch

Этот ноутбук повторяет пример «Плотное векторное представление слов» из урока 5, но
полностью на **PyTorch** (без TensorFlow/Keras). Данные те же самые, что использует
`tensorflow.keras.datasets.imdb`, — мы просто грузим их напрямую.

Учебный курс "[Программирование глубоких нейронных сетей на Python](https://openedu.ru/course/urfu/PYDNN/)".

## Набор данных IMDb movie review

[Набор данных IMDb movie review](https://ai.stanford.edu/~amaas/data/sentiment/) создан для задач определения тональности текста. Набор включает отзывы на фильмы с сайта [IMDb](https://www.imdb.com). Отзывы только явно положительные (оценка >= 7) или отрицательные (оценка <= 4), нейтральные отзывы в набор данных не включались.

Размер набора данных 50 тыс. отзывов:
- Набор данных для обучения - 25 тыс. отзывов
- Набор данных для тестирования - 25 тыс. отзывов

Количество положительных и отрицательных отзывов одинаковое.

Разметка набора данных:
- 0 - отзыв отрицательный
- 1 - отзыв положительный

С точки зрения машинного обучения это задача бинарной классификации.

Набор данных описан в статье: [Andrew L. Maas, Raymond E. Daly, Peter T. Pham, Dan Huang, Andrew Y. Ng, and Christopher Potts. (2011). Learning Word Vectors for Sentiment Analysis. The 49th Annual Meeting of the Association for Computational Linguistics (ACL 2011)](https://ai.stanford.edu/~amaas/papers/wvSent_acl2011.pdf).

Отзывы уже разбиты на последовательности номеров слов. Номера слов из словаря:
- 0 — заполнитель (padding);
- 1 — начало последовательности;
- 2 — неизвестное слово (вне словаря);
- 3 — не используется;
- 4 и выше — реальные слова, упорядоченные по частоте.

<img src="https://www.dropbox.com/s/grd17bkapocb92o/imdb_movie_reviews.png?dl=1" width="600">

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import urllib.request
import json
import os
from io import BytesIO
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
%matplotlib inline

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
print("Устройство:", device)

## Загружаем данные

Файлы `imdb.npz` и `imdb_word_index.json` — это те же архив и словарь, откуда их берёт
Keras (`tensorflow.keras.datasets.imdb`). Мы скачиваем их напрямую и сохраняем локально,
поэтому TensorFlow для этого ноутбука не нужен.

In [ ]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

# Файлы скачиваются из архива TensorFlow (тот же источник, что у keras.datasets.imdb)
GCS = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/"


In [ ]:
def download_if_absent(name, url):
    path = data_dir / name
    if not path.exists():
        print(f"Скачиваем {name} ...")
        urllib.request.urlretrieve(url, path)
    return path

npz_path = download_if_absent("imdb.npz", GCS + "imdb.npz")
word_index_path = download_if_absent("imdb_word_index.json", GCS + "imdb_word_index.json")

In [ ]:
def load_imdb(num_words=None):
    # Наборы обучения и тестирования уже в файле: 25 тыс. отзывов в каждом
    data = np.load(npz_path, allow_pickle=True)
    (x_train, y_train), (x_test, y_test) = (data["x_train"], data["y_train"]), (data["x_test"], data["y_test"])
    if num_words is not None:
        # Как делает keras.load_data: оставляем только num_words самых частотных слов
        x_train = [[int(w) for w in seq if w < num_words] for seq in x_train]
        x_test = [[int(w) for w in seq if w < num_words] for seq in x_test]
    return np.array(x_train, dtype=object), y_train, np.array(x_test, dtype=object), y_test

max_words = 10000
x_train, y_train, x_test, y_test = load_imdb(num_words=max_words)

print(f"Обучение: {len(x_train)} отзывов | Тест: {len(x_test)} отзывов")
print(f"Доля положительных в обучающем наборе: {y_train.mean():.3f}")

### Как тексты превращаются в числа

Модель работает только с числами, поэтому каждое слово заменяется своим номером в частотном словаре:

1. По всем отзывам строится словарь уникальных слов, отсортированных по частоте встречаемости: чем чаще слово, тем меньше его номер.
2. Рецензия превращается в последовательность номеров слов — каждый номер $w$ соответствует слову в словаре.
3. Ограничение словаря: `max_words = 10000` оставляет только самые частотные слова, из каждой последовательности отбрасываются токены $w \ge 10000$ (точно так же делает и Keras).

Служебные токены в начале словаря:
- 0 — заполнитель (padding);
- 1 — начало последовательности;
- 2 — неизвестное слово (вне словаря);
- 3 — не используется;
- 4 и выше — реальные слова, упорядоченные по частоте.

В следующем разделе посмотрим на отзывы глазами этих чисел.

## Просмотр данных

Рецензия (последовательность номеров слов)

In [ ]:
x_train[3]

Правильный ответ (1 - положительный отзыв)

In [ ]:
y_train[3]

## Подготовка данных для обучения

### Зачем дополнять рецензии до одинаковой длины

Связка слоёв `nn.Embedding` → `nn.Flatten` → `nn.Linear` требует вход фиксированной формы: все рецензии обязаны иметь одинаковую длину. В наборе отзывы разной длины $L_i$, поэтому:

- если $L_i > \text{maxlen} = 200$ — оставляем последние 200 слов (считаем, что конец отзыва важнее; в терминах Keras это `truncating='pre'`):
  $$ \tilde{x}_i = [x_{i,\,L_i-200}, \, \dots , \, x_{i,\,L_i}] $$
- если $L_i < 200$ — дописываем нулевые заполнители в конец (`padding='post'`):
  $$ \tilde{x}_i = [\,x_{i,1},\, \dots ,\, x_{i,L_i},\, \underbrace{0,\, \dots ,\, 0}_{200 - L_i}\,] $$

После дополнения каждая рецензия имеет форму `(200,)`, а весь набор — `(N, 200)`. Нулевые позиции почти не влияют на результат: вектор нулевой строки таблицы встраивания сеть по сути игнорирует.

In [ ]:
maxlen = 200

# Аналог keras pad_sequences(padding='post', truncating='pre'):
#   - нули добавляем после конца рецензии (padding='post');
#   - слишком длинные рецензии укорачиваем, оставляя хвост (truncating='pre').
def pad_sequences_post(seqs, maxlen):
    out = []
    for seq in seqs:
        seq = list(seq)
        if len(seq) > maxlen:
            seq = seq[-maxlen:]
        if len(seq) < maxlen:
            seq = seq + [0] * (maxlen - len(seq))
        out.append(seq)
    return np.array(out, dtype=np.int64)

x_train = pad_sequences_post(x_train, maxlen)
x_test = pad_sequences_post(x_test, maxlen)

# Превращаем в тензоры PyTorch
x_train = torch.from_numpy(x_train)
y_train = torch.from_numpy(y_train.astype(np.float32))
x_test = torch.from_numpy(x_test)
y_test = torch.from_numpy(y_test.astype(np.float32))

In [ ]:
x_train[1]

In [ ]:
y_train[1]

## Создаем нейронную сеть

Модель на PyTorch, повторяет архитектуру из версии на TensorFlow:

- `Embedding(10000, 2)` — обучаемая таблица из 10 тыс. слов, каждое слово - вектор из 2 чисел;
- `Dropout(0.25)` — регуляризация;
- `Flatten()` — вытягиваем всю рецензию в один вектор;
- `Linear(200*2, 1)` — один выходной нейрон (логит тональности).

Выход - **логит** (число до сигмоиды), поэтому считаем потери через `BCEWithLogitsLoss`.

### Математика модели

Разберём, что делает каждый из четырёх слоёв и какие формулы за ними стоят.

**1. Слой встраивания (Embedding).** `nn.Embedding(max_words, 2)` хранит таблицу весов $E \in \mathbb{R}^{V \times d}$, где $V = 10000$ — размер словаря, $d = 2$ — размерность вектора слова. Слово с номером $w$ заменяется строкой таблицы:

$$ v_w = E[w], \qquad E \in \mathbb{R}^{10000 \times 2} $$

Математически это то же самое, что умножить one-hot вектор $o_w$ (единица в позиции $w$, нули в остальных) на матрицу $E$:

$$ v_w = o_w^{\top} E $$

Обе координаты $v_w^{(0)}$ и $v_w^{(1)}$ **обучаются** вместе со всей сетью: по градиенту функции потерь обновляются только те строки таблицы, слова которых реально встретились в рецензиях пакета.

**2. Dropout.** `nn.Dropout(0.25)` во время обучения с вероятностью $p = 0.25$ «выключает» случайную долю элементов и масштабирует оставшиеся, чтобы масштаб сигнала сохранился:

$$ \tilde{v}_i = \frac{m_i \cdot v_i}{1 - p}, \qquad m_i \sim \mathrm{Bernoulli}(1 - p) $$

Это регуляризация: сеть не может опереться на отдельную координату, ведь её может «занулить» dropout. При проверке и тестировании слой работает как тождество.

**3. Flatten.** После встраивания пакет имеет форму `(batch, 200, 2)` (200 слов, у каждого вектор из 2 чисел). `nn.Flatten()` вытягивает его в один плоский вектор из $200 \cdot 2 = 400$ элементов:

$$ x \in \mathbb{R}^{200 \times 2} \;\longrightarrow\; \mathrm{vec}(x) \in \mathbb{R}^{400} $$

**4. Полносвязный слой (Linear).** `nn.Linear(400, 1)` вычисляет взвешенную сумму признаков со смещением — одно число, **логит**:

$$ z = W^{\top} x + b, \qquad W \in \mathbb{R}^{400},\; b \in \mathbb{R} $$

**5. Сигмоида на выходе.** В сети сигмоиды нет (об этом ниже), но предсказание — вероятность положительного отзыва — получается из логита именно через неё:

$$ \hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}} \in (0, 1) $$

Правило классификации: отзыв положительный ($y = 1$), если $\hat{y} \ge 0.5$, что эквивалентно $z \ge 0$.

In [ ]:
model = nn.Sequential(
    nn.Embedding(max_words, 2),              # таблица "слово -> вектор из 2 чисел"
    nn.Dropout(0.25),
    nn.Flatten(),
    nn.Linear(maxlen * 2, 1)                 # один выходной нейрон (без сигмоиды!)
).to(device)

print(model)
print(f"Количество параметров: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.BCEWithLogitsLoss()

### Функция потерь — бинарная перекрёстная энтропия

Пусть $y \in \{0, 1\}$ — истинная метка отзыва, $\hat{y}$ — предсказанная вероятность положительного класса. Логистическая (бинарная перекрёстная) энтропия по одному примеру:

$$ L(y, \hat{y}) = -\Big[\, y \ln \hat{y} + (1 - y) \ln (1 - \hat{y}) \,\Big] $$

По пакету из $N$ примеров берём среднее и минимизируем его градиентным спуском:

$$ \mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \Big[\, y_i \ln \hat{y}_i + (1 - y_i) \ln (1 - \hat{y}_i) \,\Big] $$

Свойство: чем увереннее модель ошибается (например, $y_i = 1$, а $\hat{y}_i \to 0$), тем $\ln \hat{y}_i \to -\infty$ и штраф сильнее — именно это заставляет сеть учиться.

**Почему потери считают по логитам.** `nn.BCEWithLogitsLoss()` принимает на вход логиты $z$, а не вероятности. Это объединение сигмоиды и логарифма в одну численно устойчивую формулу:

$$ L = \begin{cases} \ln\big(1 + e^{-z}\big), & y = 1 \\[4pt] \ln\big(1 + e^{z}\big), & y = 0 \end{cases} $$

При $z \to -\infty$ напрямую считать $\ln(1 / (1 + e^{-z}))$ нельзя (потеря младших разрядов, переполнение), а формула выше не даёт промежуточных значений $\sigma(z)$ и считается стабильно. Поэтому сигмоиду в сеть отдельно не встраивают: для предсказаний её применяют на этапе оценки.

## Обучаем нейронную сеть

### Оптимизатор Adam — как обновляются веса

Потери минимизирует оптимизатор **Adam** (Adaptive Moment Estimation). Adam поддерживает скользящие средние градиента $m_t$ и его квадрата $v_t$:

$$ m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t, \qquad v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2 $$

где $g_t = \nabla_\theta \mathcal{L}$ — градиент функции потерь по параметрам $\theta$. Так как средние стартуют с нуля, на первых шагах их смещение исправляется (bias correction):

$$ \hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \qquad \hat{v}_t = \frac{v_t}{1 - \beta_2^t} $$

Шаг обновления параметров:

$$ \theta_t = \theta_{t-1} - \alpha \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \varepsilon} $$

Смысл: направление задаёт сглаженный градиент $\hat{m}_t$ (момент, как при качении шара), а размер шага автоматически масштабируется — на «крутых» направлениях с большим $\hat{v}_t$ шаг меньше, на «пологих» больше. Стандартные параметры: $\alpha = 0.001$, $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\varepsilon = 10^{-8}$.

Обучение идёт по **мини-пакетам**: за шаг градиент считается по 128 рецензиям, а не по всей выборке (это быстрее и устойчивее). Отдельно на 10% обучающих примеров (переменная `val_idx`) измеряется точность во время обучения — так видно, не переобучается ли сеть. Схема: обучение (`model.train`, включён dropout) → проверка (`model.eval`, dropout выключен, градиенты не считаются).

In [ ]:
epochs = 15
batch_size = 128

# Отделяем 10% обучающих данных для проверки во время обучения
train_idx, val_idx = train_test_split(np.arange(x_train.size(0)), test_size=0.1, random_state=42)
train_loader = DataLoader(TensorDataset(x_train[train_idx], y_train[train_idx]),
                          batch_size=batch_size, shuffle=True, num_workers=2)

history = {"loss": [], "accuracy": [], "val_accuracy": []}
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb).squeeze(1)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == yb).sum().item()
        total += xb.size(0)

    # Точность на проверочном наборе
    model.eval()
    with torch.no_grad():
        val_correct = 0
        for xb, yb in DataLoader(TensorDataset(x_train[val_idx], y_train[val_idx]),
                                 batch_size=512, shuffle=False):
            xb, yb = xb.to(device), yb.to(device)
            preds = (torch.sigmoid(model(xb).squeeze(1)) >= 0.5).float()
            val_correct += (preds == yb).sum().item()
        val_acc = val_correct / len(val_idx)

    acc = correct / total
    history["loss"].append(total_loss / total)
    history["accuracy"].append(acc)
    history["val_accuracy"].append(val_acc)
    print(f"Эпоха {epoch:2d} | loss {history['loss'][-1]:.4f} | "
          f"accuracy {acc:.3f} | val_accuracy {val_acc:.3f}")

In [ ]:
plt.plot(history['accuracy'],
         label='Доля правильных ответов на обучающем наборе')
plt.plot(history['val_accuracy'],
         label='Доля правильных ответов на проверочном наборе')
plt.xlabel('Эпоха обучения')
plt.ylabel('Доля правильных ответов')
plt.legend()
plt.show()

## Проверяем работу сети на тестовом наборе данных

Качество на отложенном тестовом наборе (25 тыс. отзывов, не участвовавших в обучении) измеряется метрикой **accuracy** — долей верно предсказанных меток:

$$ \mathrm{Acc} = \frac{1}{N_{\mathrm{test}}} \sum_{i=1}^{N_{\mathrm{test}}} \mathbb{I}\big[\hat{y}_i = y_i\big] $$

где $\mathbb{I}[\cdot]$ — индикатор совпадения предсказанного класса с истинной меткой. Проверку делаем в блоке `torch.no_grad()` (градиенты не нужны — это экономит память) и пакетами по 512 примеров.

In [ ]:
model.eval()
test_correct = 0
test_loss = 0.0
with torch.no_grad():
    for xb, yb in DataLoader(TensorDataset(x_test, y_test), batch_size=512):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb).squeeze(1)
        test_loss += loss_fn(logits, yb).item() * xb.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        test_correct += (preds == yb).sum().item()

scores = (test_loss / x_test.size(0), test_correct / x_test.size(0))
print(f"Потери на тестовом наборе: {scores[0]:.4f}")
print(f"Точность на тестовом наборе: {scores[1]:.4f}")
scores

## Исследуем обученное плотное векторное представление слов

**Получаем матрицу плотных векторных представлений слов.** Первый слой модели - `nn.Embedding`; его веса и есть таблица слово -> вектор. В нашем случае вектор каждого слова состоит всего из 2 чисел, поэтому его можно нарисовать на плоскости.

### Что такое обученное векторное представление слова

После обучения `embedding_matrix` — это таблица, в которой **каждая строка — вектор своего слова** из 2 чисел. Такие векторы называют *встраиванием* (embedding), или плотным векторным представлением слова.

Смысл подхода (дистрибуционная гипотеза): значение слова определяется текстами, где оно встречается, поэтому сеть учит координаты не «в лоб», а так, чтобы они были полезны для ответа. Наша сеть по сумме векторов всех слов рецензии должна выдать её тональность, значит «положительные» слова вынуждены оказаться в одной области плоскости, а «отрицательные» — в другой, иначе не разделить классы.

Благодаря размерности $d = 2$ каждое слово можно нарисовать точкой и глазами увидеть, как распределились слова по смыслу.

In [ ]:
embedding_matrix = model[0].weight.detach().cpu().numpy()
embedding_matrix.shape

In [ ]:
embedding_matrix[:5]

**Загружаем словарь с номерами слов** (тот же JSON-файл, что использует Keras)

In [ ]:
with open(word_index_path) as f:
    word_index_org = json.load(f)

# Первые записи словаря
list(word_index_org.items())[:5]

Дополняем словарь служебными символами. Номера слов из JSON нужно сдвинуть на 3,
чтобы они совпали с номерами, под которыми слова сидят в таблице векторов.

In [ ]:
word_index = dict()
for word, number in word_index_org.items():
    word_index[word] = number + 3
word_index["<Заполнитель>"] = 0
word_index["<Начало последовательности>"] = 1
word_index["<Неизвестное слово>"] = 2
word_index["<Не используется>"] = 3

**Ищем векторы для слов**

In [ ]:
word = 'good'
word_number = word_index[word]
print('Номер слова', word_number)
print('Вектор для слова', embedding_matrix[word_number])

## Сохраняем обученные плотные векторные представления в файл

**Составляем реверсивный словарь токенов (слов)**

In [ ]:
reverse_word_index = dict()
for key, value in word_index.items():
    reverse_word_index[value] = key

**Записываем плотные векторные представления в файл**

In [ ]:
filename = 'data/imdb_embeddings_pytorch.csv'
print('Сохраняем векторы слов в файл', filename)

In [ ]:
with open(filename, 'w') as f:
    for word_num in range(max_words):
        word = reverse_word_index[word_num]
        vec = embedding_matrix[word_num]
        try:
            f.write(word + ",")
            f.write(','.join([str(x) for x in vec]) + "\n")
        except UnicodeEncodeError:
            pass
print('Готово, записано строк:', max_words)

## Визуализация плотных векторных представлений слов

Так как вектор каждого слова состоит из 2 чисел, нарисуем все 10 тысяч слов точками на плоскости.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(embedding_matrix[:, 0], embedding_matrix[:, 1], s=6, alpha=0.4)
plt.xlabel('Первая координата вектора слова')
plt.ylabel('Вторая координата вектора слова')
plt.title('Векторы всех слов из словаря')
plt.show()

Выбираем коды слов, по которым можно определить тональность отзыва

In [ ]:
review = ['brilliant', 'fantastic', 'amazing', 'good',
          'bad', 'awful', 'crap', 'terrible', 'trash']
enc_review = []
for word in review:
    enc_review.append(word_index[word])
enc_review

Получаем векторное представление интересующих нас слов

In [ ]:
review_vectors = embedding_matrix[enc_review]
review_vectors

Визуализация обученного плотного векторного представления слов, по которым можно определить эмоциональную окраску текста

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(review_vectors[:, 0], review_vectors[:, 1])
for i, txt in enumerate(review):
    plt.annotate(txt, (review_vectors[i, 0], review_vectors[i, 1]), fontsize=12)
plt.xlabel('Первая координата вектора слова')
plt.ylabel('Вторая координата вектора слова')
plt.title('Векторы слов, передающих эмоциональную окраску отзыва')
plt.show()